In [87]:
conda install numpy opencv matplotlib

3 channel Terms of Service accepted
Channels:
 - defaults
Platform: win-64
Solving environment: done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.




==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.3.1

Please update conda by running

    $ conda update -n base -c defaults conda




In [88]:
pip install mediapipe # media pipe is basically google's computer vision based API package

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [89]:
# Importing the packages 
import numpy as np
import cv2
import mediapipe as mp 
import time
import math
import winsound 
from winsound import Beep

In [90]:
'''Note - There are some very obvious redundancy checks that I didn't put in  partially because those case scenarios are very hard to come across in my implementation, 
if this code was implemented in the field its very obvious that the implementation would be stronger (this is just a fun MVP)'''

"Note - There are some very obvious redundancy checks that I didn't put in  partially because those case scenarios are very hard to come across in my implementation, \nif this code was implemented in the field its very obvious that the implementation would be stronger (this is just a fun MVP)"

In [91]:

# Our main variable that defines our camera source and purpose
capture = cv2.VideoCapture(0) 

# Importing the task file (pretrained)
model_path = 'face_landmarker.task'

# Importing the second task_file
model_path_hand = 'hand_landmarker.task'


In [92]:

# Importing the necessary classes 
''' AI generated description : Think of them as a small assembly line:

BaseOptions → tells the detector where the model is and which device to use.
FaceLandmarkerOptions → configures the detector using BaseOptions plus other parameters (faces, confidence, running mode).
VisionRunningMode → tells the detector how to process frames (single image, video, live).
FaceLandmarker → the detector object you create using FaceLandmarkerOptions.
FaceLandmarkerResult → what you get after calling .process(frame) on FaceLandmarker.'''

#Face tracing classes
BaseOptions = mp.tasks.BaseOptions 
VisionRunningMode = mp.tasks.vision.RunningMode
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
FaceLandmarkerResult = mp.tasks.vision.FaceLandmarkerResult


#Hand tracing classes 

HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
HandLandmarkerResult = mp.tasks.vision.HandLandmarkerResult


In [93]:
# defining the function that's gonna help us draw on our face

def draw_face_landmarks_on_stream(rgb_frame, detection_result):
    annotated_frame = np.copy(rgb_frame)
    height, width, _ = annotated_frame.shape

    for face_landmarks in detection_result.face_landmarks:
        for landmark in face_landmarks:
            x = int(landmark.x * width)
            y = int(landmark.y * height)
            cv2.circle(rgb_frame, (x, y), 1, (0, 255, 0), -1)  #draw each landmark dot



In [94]:
# defining the function that's gonna help us draw on our hand(THe only difference from the face landmarks being the hand landmarks list )

def draw_hand_landmarks_on_stream(rgb_frame, hand_result):
    annotated_frame = np.copy(rgb_frame)
    height, width, _ = annotated_frame.shape

    for hand_landmarks in hand_result.hand_landmarks:
        # draw points
        for landmark in hand_landmarks:
            x = int(landmark.x * width)
            y = int(landmark.y * height)
            cv2.circle(rgb_frame, (x, y), 3, (255, 0, 0), -1)


In [95]:
base_options = BaseOptions(
    model_asset_path="face_landmarker.task"
) 

face_result = None
#defining a final callback function to use in our code under options
def trace_result_face(result: FaceLandmarkerResult,output_image: mp.Image,timestamp_ms: int):
    global face_result
    face_result = result

    
# defining extra features such as our running mode and feeding our base options class further into a "pipeline"( of sorts)
options = FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=VisionRunningMode.LIVE_STREAM, # utilising the proper running mode for our data 
    num_faces = 1,
    min_face_detection_confidence = 0.5,
    min_face_presence_confidence = 0.5,
    result_callback = trace_result_face)

# creating an actual landmarker object 
landmarker_face = FaceLandmarker.create_from_options(options)

In [96]:

# HAND options and landmarking

base_options_hand = BaseOptions(
    model_asset_path="hand_landmarker.task"
) 

hand_result = None
def trace_result_hand(result: HandLandmarkerResult,output_image: mp.Image,timestamp_ms: int):
    global hand_result
    hand_result = result 


options_hand = HandLandmarkerOptions(
    base_options=base_options_hand,
    running_mode=VisionRunningMode.LIVE_STREAM, # utilising the proper running mode for our data 
    num_hands = 2,
    min_hand_detection_confidence = 0.5,
    min_hand_presence_confidence = 0.5,
    result_callback = trace_result_hand)

landmarker_hands = HandLandmarker.create_from_options(options_hand) 

In [ ]:
'''pseudo code for my nailbiting detector 

important Point list hand = 
important point list face = 


If euclidean distance of any of the points in list hand is less than a minimum from list face
    play sound


Its also safe to say it could be really fun to experiment with the sounds, especially recording your own funny sounds
'''




important_hand_points_list = [3, 4, 7, 8, 11, 12, 15, 16, 19, 20]
important_face_points_list = [91, 181, 84, 17, 314, 405, 321, 375, 80, 81, 82, 13, 312, 311, 310]

PROXIMITY_THRESHOLD = 10

# Acknowledgememt- Although the high level logic of this euclidean distance calculator was something that I came up with, the code itself is AI generated 

def check_proximity(face_result, hand_result, frame_width, frame_height):
    if not face_result or not hand_result:
        return False
    if not face_result.face_landmarks or not hand_result.hand_landmarks:
        return False

    face_landmarks = face_result.face_landmarks[0]
    face_points = [
        (int(face_landmarks[i].x * frame_width), int(face_landmarks[i].y * frame_height))
        for i in important_face_points_list
    ]

    for hand_landmarks in hand_result.hand_landmarks:
        hand_points = [
            (int(hand_landmarks[i].x * frame_width), int(hand_landmarks[i].y * frame_height))
            for i in important_hand_points_list
        ]

        for h_point in hand_points:
            for f_point in face_points:
                if math.dist(h_point, f_point) < PROXIMITY_THRESHOLD:
                    return True

    return False





In [98]:
# The main frame loop ( the idea is that capturing frames in a loop will yield in a video )
while True : 

    frame_timestamp_ms = int(time.time() * 1000) # defining our real time 

    checking_bool, frame = capture.read() # ret is a boolean value to see if the frame was capture successfully and frame is the actual image capture

    if not checking_bool:
        print("Not able to capture the frame successfully")
        break

    rgb_frame = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB) # changing the colour of our input from BGR to RGB cause mediapipe takes RGB

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)# Convert the frame received from OpenCV to a MediaPipe’s Image object.

    landmarker_face.detect_async(mp_image, frame_timestamp_ms)
    landmarker_hands.detect_async(mp_image, frame_timestamp_ms)
    
    display = np.copy(rgb_frame)

    if face_result:
        draw_face_landmarks_on_stream(display, face_result)
    if hand_result:
        draw_hand_landmarks_on_stream(display, hand_result)
    
    if face_result or hand_result:
        bgr_output = cv2.cvtColor(display, cv2.COLOR_RGB2BGR)  # convert back to BGR 
        cv2.imshow('Landmarks', bgr_output)
    else :
        cv2.imshow('Landmarks', frame) # show plain frame until callback fires
    
    

    height, width, _ = display.shape
    if check_proximity(face_result, hand_result, width, height):
        winsound.Beep(1000, 1000)



    

   

    if cv2.waitKey(33) == ord('c') : # wait for this much time in milliseconds before each frame , if c is pressed then releases the video and closes all windows, aiming for 30 fps
        break

landmarker_face.close()# for face 
landmarker_hands.close()
capture.release()
cv2.destroyAllWindows() # A very dramatic sounding function to close the captured windows






In [99]:
# I tried my best to go through the documentation as much as I could and use AI as less as possible to make sure I balance my learning and implementation skills


# AI usage - https://chatgpt.com/share/69ce5a9a-7504-8321-aeb8-baacdf0182a6 ,https://chatgpt.com/c/69d0bc04-fbb4-8322-8264-d83d3f3c7dd1, https://claude.ai/chat/eb235ade-5185-4ae5-9550-449da1ef54dc 

''' Internet resources used :
https://docs.opencv.org/4.13.0/
https://medium.com/@alionurulker/live-stream-on-any-camera-using-opencv-and-python-e18d4de6fad7 
https://docs.opencv.org/4.x/dd/d43/tutorial_py_video_display.html 
https://youtu.be/RnQy7U5Eu94?si=06W-LhxrypmYoJu5
https://www.geeksforgeeks.org/python/python-opencv-waitkey-function 
https://youtu.be/hV5S4iQhNkI?si=aZB8sLVnCiwGMbkV 
https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/python 
https://youtu.be/rAS17tDYeA0?si=pU7OKbfWT9PtvrfB # ditched this resource because it was building off of a legacy API but utilised it regardless in my understanding of the various functionalities provided by mediapipe
https://youtu.be/loEZnF7Z-Zk?si=VDWIOGi6ZLPP0kdS 
https://storage.googleapis.com/mediapipe-assets/documentation/mediapipe_face_landmark_fullsize.png # Used this to gauge which landmark points corresponded to which part of the face 
https://docs.python.org/3/library/winsound.html


'''

' Internet resources used :\nhttps://docs.opencv.org/4.13.0/\nhttps://medium.com/@alionurulker/live-stream-on-any-camera-using-opencv-and-python-e18d4de6fad7 \nhttps://docs.opencv.org/4.x/dd/d43/tutorial_py_video_display.html \nhttps://youtu.be/RnQy7U5Eu94?si=06W-LhxrypmYoJu5\nhttps://www.geeksforgeeks.org/python/python-opencv-waitkey-function \nhttps://youtu.be/hV5S4iQhNkI?si=aZB8sLVnCiwGMbkV \nhttps://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/python \nhttps://youtu.be/rAS17tDYeA0?si=pU7OKbfWT9PtvrfB # ditched this resource because it was building off of a legacy API but utilised it regardless in my understanding of the various functionalities provided by mediapipe\nhttps://youtu.be/loEZnF7Z-Zk?si=VDWIOGi6ZLPP0kdS \nhttps://storage.googleapis.com/mediapipe-assets/documentation/mediapipe_face_landmark_fullsize.png # Used this to gauge which landmark points corresponded to which part of the face \nhttps://docs.python.org/3/library/winsound.html\n\n\n'